# LAB 08 - TravelOps
## Notebook: 00_seed_raw_data

Purpose:
Seeds selected `samples.wanderbricks` source tables into the Terraform-owned raw landing Volume.

Business purpose:
Creates repeatable raw travel booking data so CI/CD can prove the same application promotes from DEV to PROD.

Technical purpose:
Verifies the DAB target schema and Terraform-owned raw Volume, then additively writes a new content-fingerprinted Parquet file per source table for Auto Loader, without deleting any existing raw file.

Inputs:
- samples.wanderbricks.bookings
- samples.wanderbricks.booking_updates
- samples.wanderbricks.payments
- samples.wanderbricks.users
- samples.wanderbricks.properties
- samples.wanderbricks.reviews
- samples.wanderbricks.destinations

Outputs:
- /Volumes/<raw_volume_catalog>/<raw_volume_schema>/<raw_volume>/raw/<source_table>/

Tables/files affected:
Only raw Parquet folders under `raw/<source_table>/` are written to, and only additively: this notebook never deletes an existing file there. Schema and Volume metadata are owned by Terraform.

Environment variables/widgets used:
`target_catalog`, `target_schema`, `raw_volume_catalog`, `raw_volume_schema`, `raw_volume_name`, `raw_volume_type`, `raw_volume_storage_location`, `source_catalog`, `source_schema`, `seed_limit`.

Creates/modifies data:
Yes, but only additively. A rerun whose sampled content is byte-for-byte identical to the last run (same `seed_limit` and unchanged `samples.wanderbricks` content) writes nothing at all. A rerun whose content differs writes one new file; it never deletes the previous one.

Dependencies/prerequisites:
The DAB target schema must exist, Terraform must have already created the raw Volume, and the deployer must have read access to `samples.wanderbricks`.

Expected result:
Each configured source table has a raw Parquet file named from `seed_limit` and a content fingerprint in the target Volume. `bookings`, `users`, `properties` and `destinations` are sampled independently by row order. `booking_updates`, `payments` and `reviews` are booking-scoped event tables, so they are instead filtered to the exact `booking_id` set sampled from `bookings`, keeping every related event for a sampled booking instead of an independent, referentially-inconsistent row-count cap.

Idempotency note:
Ingestion idempotency is handled at the source, not just downstream. Each table's sampled content is fingerprinted (`_content_fingerprint`, an order-independent XOR of a 64-bit hash per row) and written to a path derived from both `seed_limit` and that fingerprint (`_seed_file_name`), instead of letting Spark allocate a fresh, uniquely-named file on every write. Before writing, the notebook checks whether a file at that exact path already exists (`_existing_seed_path`) and skips the write entirely if so -- this is the common case (a routine rerun with unchanged `seed_limit` and unchanged upstream data), and it touches no files at all. When the content genuinely changes (a `seed_limit` change, or `samples.wanderbricks` itself changing upstream), the fingerprint changes, a new file is written, and Auto Loader picks it up as new -- this is based on Auto Loader's documented default `cloudFiles.allowOverwrites=false`, which does not reprocess a path it has already ingested; it has not been empirically validated by an actual run in this repository (see `evidence/lab08_production_remediation_plan.md` for the validation procedure).

This notebook never deletes an existing raw file to make room for a new one: the previous implementation deleted the target directory before writing its replacement, which would have left the raw Volume with no valid input at all if the write or move step failed partway through. A superseded content-version file (from a genuine `seed_limit`/source change, not a routine rerun) is deliberately left in place rather than cleaned up automatically; see the remediation plan for that trade-off and how to clean it up under explicit approval. This also means historical Bronze duplicates already accumulated by the previous overwrite-based implementation are not removed by this fix -- `pipeline/silver.py` still deduplicates `payments_silver` and `reviews_silver` on composite business keys, and `properties_silver`/`destinations_silver`/`users_silver` on their primary key, as a defense-in-depth backstop for that residual case.

Failure behavior:
The notebook fails fast when required parameters are missing, when the Terraform-owned raw Volume is absent, or when source/target privileges are insufficient.

Environment classification:
Safe for personal_dev and personal_prod runs. Azure PROD job execution is intentionally deferred until explicitly authorized.


### Step 1 - Resolve target configuration

This cell reads Databricks widgets supplied by the Bundle job. It validates required values before any write occurs so configuration errors fail before partial data is created.

In [ ]:
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("target_schema", "")
dbutils.widgets.text("raw_volume_catalog", "")
dbutils.widgets.text("raw_volume_schema", "")
dbutils.widgets.text("raw_volume_name", "lab08_dev_travelops_raw")
dbutils.widgets.text("raw_volume_type", "MANAGED")
dbutils.widgets.text("raw_volume_storage_location", "")
dbutils.widgets.text("source_catalog", "samples")
dbutils.widgets.text("source_schema", "wanderbricks")
dbutils.widgets.text("seed_limit", "25000")

target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
raw_volume_catalog = dbutils.widgets.get("raw_volume_catalog")
raw_volume_schema = dbutils.widgets.get("raw_volume_schema")
raw_volume_name = dbutils.widgets.get("raw_volume_name")
raw_volume_type = dbutils.widgets.get("raw_volume_type").upper()
raw_volume_storage_location = dbutils.widgets.get("raw_volume_storage_location")
source_catalog = dbutils.widgets.get("source_catalog")
source_schema = dbutils.widgets.get("source_schema")
seed_limit = int(dbutils.widgets.get("seed_limit"))

required = {
    "target_catalog": target_catalog,
    "target_schema": target_schema,
    "raw_volume_catalog": raw_volume_catalog,
    "raw_volume_schema": raw_volume_schema,
    "raw_volume_name": raw_volume_name,
    "source_catalog": source_catalog,
    "source_schema": source_schema,
}
missing = [name for name, value in required.items() if not value]
if missing:
    raise ValueError(f"Missing required widgets: {missing}")
if raw_volume_type == "EXTERNAL" and not raw_volume_storage_location:
    raise ValueError("External raw volume requires raw_volume_storage_location")


### Step 2 - Verify target schema and Terraform-owned raw Volume

This cell performs read-only metadata checks. It intentionally does not create or alter the raw Volume schema or raw Volume because Terraform owns that layer; the application target schema is checked separately.

In [ ]:
target_schema_exists = spark.sql(f"SHOW SCHEMAS IN `{target_catalog}` LIKE '{target_schema}'").count() == 1
if not target_schema_exists:
    raise ValueError(f"Required DAB target schema does not exist: {target_catalog}.{target_schema}")

raw_schema_exists = spark.sql(f"SHOW SCHEMAS IN `{raw_volume_catalog}` LIKE '{raw_volume_schema}'").count() == 1
if not raw_schema_exists:
    raise ValueError(f"Required Terraform-referenced raw schema does not exist: {raw_volume_catalog}.{raw_volume_schema}")

volume_rows = spark.sql(f"SHOW VOLUMES IN `{raw_volume_catalog}`.`{raw_volume_schema}` LIKE '{raw_volume_name}'").collect()
if len(volume_rows) != 1:
    raise ValueError(f"Required Terraform-owned raw Volume is missing: {raw_volume_catalog}.{raw_volume_schema}.{raw_volume_name}")

print(f"Verified DAB target schema: {target_catalog}.{target_schema}")
print(f"Verified Terraform-owned raw Volume: {raw_volume_catalog}.{raw_volume_schema}.{raw_volume_name}")


### Step 3 - Seed deterministic raw Parquet folders

This cell reads the discovered Wanderbricks tables and overwrites deterministic target folders. Overwrite mode prevents uncontrolled duplicate files across reruns while keeping the raw contract file-based for Auto Loader.

In [ ]:
from pyspark.sql import functions as F

source_tables = [
    "bookings",
    "booking_updates",
    "payments",
    "users",
    "properties",
    "reviews",
    "destinations",
]

# booking_updates, payments and reviews reference booking_id. Sampling each
# of these independently by row order (as bookings/users/properties/
# destinations still are) breaks that relationship: a booking's related
# payment or update rows can fall outside a same-sized but differently
# ordered independent sample, which previously produced "missing payment"
# rows caused purely by sampling rather than by the source data. Sampling
# bookings first and then filtering these three tables to that exact
# booking_id set keeps every related event for a sampled booking.
BOOKING_SCOPED_TABLES = {"booking_updates", "payments", "reviews"}

bookings_source = f"`{source_catalog}`.`{source_schema}`.`bookings`"
bookings_df = spark.table(bookings_source)
bookings_sample_df = bookings_df.orderBy(*bookings_df.columns[:1]).limit(seed_limit)
sampled_booking_ids_df = bookings_sample_df.select("booking_id")


def _content_fingerprint(df) -> str:
    """Order-independent content fingerprint for a DataFrame.

    XOR-aggregating a 64-bit hash of every row gives a fingerprint that
    changes whenever the row content changes -- including an upstream
    samples.wanderbricks change that leaves seed_limit untouched -- while
    staying stable when the exact same rows are reseeded. XOR (not SUM) is
    used because it cannot overflow and is commutative/associative, so row
    order and partitioning never affect the result. This is the actual
    content-version identifier; seed_limit alone is not, since two runs
    with the same seed_limit could still see different upstream data.
    """

    row_hash_df = df.select(F.xxhash64(*df.columns).alias("row_hash"))
    result = row_hash_df.agg(F.expr("bit_xor(row_hash)").alias("fp")).collect()[0]
    fingerprint = result["fp"] if result["fp"] is not None else 0
    return format(fingerprint & 0xFFFFFFFFFFFFFFFF, "016x")


def _seed_file_name(current_seed_limit: int, fingerprint: str) -> str:
    """Deterministic Parquet file name for this run's raw seed content.

    Encodes both seed_limit and a content fingerprint (see
    _content_fingerprint) so the path changes whenever the actual sampled
    rows change for any reason -- a seed_limit config change, or the
    upstream samples.wanderbricks source itself changing while seed_limit
    stays the same. seed_limit alone was not a safe content version: Auto
    Loader's default cloudFiles.allowOverwrites=false does not reprocess an
    unchanged path, so treating an unchanged path as "unchanged content"
    requires a real content fingerprint, not just a config value that could
    stay the same while the underlying data changes.
    """

    return f"seed-{current_seed_limit}-{fingerprint}.snappy.parquet"


def _existing_seed_path(target_path: str, file_name: str):
    """Return the full path of file_name under target_path if present, else None.

    Never deletes or modifies anything; a missing target_path (first-ever
    seed for this table) is treated as "no existing content" rather than an
    error.
    """

    try:
        existing_names = {f.name for f in dbutils.fs.ls(target_path)}
    except Exception:
        return None
    return f"{target_path}/{file_name}" if file_name in existing_names else None


def _write_new_seed_content(df, target_path: str, staging_root: str, table_name: str, file_name: str) -> None:
    """Write df as a new, additively-named Parquet file under target_path.

    This never deletes or overwrites any existing file under target_path.
    The new content lands at a path derived from _seed_file_name, which by
    construction differs from every prior seed file for this table (since
    it changed content, or it would have matched _existing_seed_path and
    this function would not have been called). If a prior content
    version's seed file is still present, it is deliberately left in place:
    deleting the previous file before confirming the new one is fully
    written and moved into place would risk leaving target_path with no
    valid raw input at all if this step failed partway through (for
    example if the job were killed between deleting the old file and
    moving the new one in). Any such leftover superseded file is a
    known, low-urgency cleanup item -- see
    evidence/lab08_production_remediation_plan.md -- not something this
    notebook deletes automatically.
    """

    staging_path = f"{staging_root}/{table_name}"
    try:
        dbutils.fs.rm(staging_path, recurse=True)
    except Exception:
        pass

    df.coalesce(1).write.mode("overwrite").format("parquet").save(staging_path)
    part_files = [f.path for f in dbutils.fs.ls(staging_path) if f.name.startswith("part-")]
    if len(part_files) != 1:
        raise AssertionError(
            f"Expected exactly one staged part file for {table_name}, found {len(part_files)}: {part_files}"
        )

    final_path = f"{target_path}/{file_name}"
    dbutils.fs.mv(part_files[0], final_path)
    dbutils.fs.rm(staging_path, recurse=True)


staging_root = f"/Volumes/{raw_volume_catalog}/{raw_volume_schema}/{raw_volume_name}/_seed_staging"

seed_summary = []
for table_name in source_tables:
    source_table = f"`{source_catalog}`.`{source_schema}`.`{table_name}`"
    target_path = f"/Volumes/{raw_volume_catalog}/{raw_volume_schema}/{raw_volume_name}/raw/{table_name}"
    if table_name == "bookings":
        df = bookings_sample_df
    elif table_name in BOOKING_SCOPED_TABLES:
        df = spark.table(source_table).join(sampled_booking_ids_df, "booking_id", "left_semi")
    else:
        source_df = spark.table(source_table)
        df = source_df.orderBy(*source_df.columns[:1]).limit(seed_limit)

    row_count = df.count()
    fingerprint = _content_fingerprint(df)
    file_name = _seed_file_name(seed_limit, fingerprint)
    existing_path = _existing_seed_path(target_path, file_name)
    if existing_path is not None:
        seed_status = "unchanged (write skipped)"
        final_path = existing_path
    else:
        _write_new_seed_content(df, target_path, staging_root, table_name, file_name)
        seed_status = "written (new content version)"
        final_path = f"{target_path}/{file_name}"

    seed_summary.append((table_name, row_count, final_path, seed_status))

display(
    spark.createDataFrame(
        seed_summary, "table_name STRING, row_count LONG, target_path STRING, seed_status STRING"
    )
)
